In [1]:
import sys
import asyncio
import nest_asyncio

# 1. Apply the Jupyter patch (allows nested loops)
nest_asyncio.apply()

# 2. Apply the Windows Fix (Required for Playwright)
# Without this, you get 'NotImplementedError' on Windows
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

print("✅ Setup complete. Windows Policy set and Asyncio patched.")

✅ Setup complete. Windows Policy set and Asyncio patched.


In [2]:
# Cell 2: Native Async Scraper (Enhanced with Batching, Blocking, and Smart Parsing)
import asyncio
import json
import hashlib
import re
import random
import chromadb
from datetime import datetime, timedelta
from playwright.async_api import async_playwright
from playwright_stealth import stealth_async
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
CHROMA_PATH = "./joblens_db"
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
CONCURRENCY_LIMIT = 5  # Number of tabs to open at once

class JobLensScraper:
    def __init__(self):
        self.chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
        self.collection = self.chroma_client.get_or_create_collection(name="job_listings")
        print("Loading AI Model...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        
        existing_data = self.collection.get()
        self.existing_ids = set(existing_data['ids'])
        print(f"Loaded {len(self.existing_ids)} existing jobs.")

    def normalize_text(self, text):
        if not text: return ""
        return re.sub(r'[^a-z0-9]', '', text.lower())

    def generate_job_id(self, title, company):
        raw_sig = f"{self.normalize_text(title)}|{self.normalize_text(company)}"
        return hashlib.md5(raw_sig.encode()).hexdigest()

    def is_duplicate(self, job_id):
        return job_id in self.existing_ids

    def save_jobs(self, jobs):
        if not jobs: return
        new_ids, new_docs, new_metas, new_embeddings = [], [], [], []
        
        print(f"   Processing {len(jobs)} candidates for database...")
        for job in jobs:
            job_id = self.generate_job_id(job['title'], job['company'])
            if self.is_duplicate(job_id): continue 
            
            # Combine Title + Skills + Description + Location for a rich embedding
            skills_str = ", ".join(job.get('skills', []))
            text_for_embedding = (
                f"{job['title']} at {job['company']}. "
                f"Location: {job['location']}. "
                f"Skills: {skills_str}. "
                f"{job.get('description', '')}"
            )
            
            new_ids.append(job_id)
            new_docs.append(text_for_embedding)
            
            meta = {
                "source": job['source'], 
                "title": job['title'], 
                "company": job['company'],
                "location": job['location'],
                "job_page_link": job['job_page_link'],
                "apply_link": job.get('apply_link', job['job_page_link']),
                "posted_time": job.get('posted_time', str(datetime.now().date())), # Now stores real date
                
                "experience_level": job.get('experience_level', 'Not Specified'),
                "employment_type": job.get('employment_type', 'Not Specified'),
                "skills_list": ", ".join(job.get('skills', [])),
                "description_snippet": job.get('description', '')[:200],
                "json_detailed": json.dumps(job) 
            }
            new_metas.append(meta)
            new_embeddings.append(self.model.encode(text_for_embedding).tolist())
            self.existing_ids.add(job_id)

        if new_ids:
            self.collection.upsert(ids=new_ids, embeddings=new_embeddings, metadatas=new_metas, documents=new_docs)
            print(f"   ✅ Success: Added {len(new_ids)} NEW jobs to DB.")
        else:
            print("   ⚠️ All jobs were duplicates.")

# --- HELPER: TEXT & DATE PARSER ---
def parse_relative_date(text):
    """Converts '2 days ago', '1 week ago' to YYYY-MM-DD string."""
    if not text: return str(datetime.now().date())
    text = text.lower()
    today = datetime.now()
    delta = timedelta(days=0)
    
    try:
        if "hour" in text or "minute" in text or "just now" in text:
            delta = timedelta(days=0)
        elif "day" in text:
            num = int(re.search(r'\d+', text).group())
            delta = timedelta(days=num)
        elif "week" in text:
            num = int(re.search(r'\d+', text).group())
            delta = timedelta(weeks=num)
        elif "month" in text:
            num = int(re.search(r'\d+', text).group())
            delta = timedelta(days=num * 30)
        return (today - delta).strftime("%Y-%m-%d")
    except:
        return str(today.date())

def parse_description(full_text):
    lower_text = full_text.lower()
    req_start = -1
    if "requirements" in lower_text: req_start = lower_text.find("requirements")
    elif "qualifications" in lower_text: req_start = lower_text.find("qualifications")
    
    resp_start = -1
    if "responsibilities" in lower_text: resp_start = lower_text.find("responsibilities")
    elif "duties" in lower_text: resp_start = lower_text.find("duties")
    
    requirements = ""
    responsibilities = ""
    
    if req_start != -1 and resp_start != -1:
        if req_start < resp_start:
            requirements = full_text[req_start:resp_start]
            responsibilities = full_text[resp_start:]
        else:
            responsibilities = full_text[resp_start:req_start]
            requirements = full_text[req_start:]
    elif req_start != -1:
        requirements = full_text[req_start:]
    elif resp_start != -1:
        responsibilities = full_text[resp_start:]
        
    return requirements.strip(), responsibilities.strip()

# --- HELPER: BLOCK RESOURCES ---
async def block_resources(route):
    """Blocks images, fonts, and CSS to speed up loading."""
    if route.request.resource_type in ["image", "media", "font", "stylesheet"]:
        await route.abort()
    else:
        await route.continue_()

# --- HELPER: GET FULL DETAILS ---
async def get_job_details(context, link, source):
    if not link: return {}
    
    # Random short sleep to be polite but fast
    await asyncio.sleep(random.uniform(0.5, 1.5))
    
    page = await context.new_page()
    # Apply resource blocking on this new page
    await page.route("**/*", block_resources)
    
    data = {
        "description": "",          
        "requirements": "",         
        "responsibilities": "",     
        "skills": [],               
        "experience_level": "Not Specified",
        "employment_type": "Not Specified",
        "apply_link": link
    }
    
    try:
        await page.goto(link, timeout=10000) # Reduced timeout for speed
        
        if source == "Wuzzuf":
            try:
                elm = await page.wait_for_selector(".css-1uobp1k", timeout=3000)
                data["description"] = await elm.inner_text()
            except: data["description"] = "Description not found."
            
            try:
                skill_tags = await page.query_selector_all(".css-158idm4, a.css-o171kl") 
                data["skills"] = [(await t.inner_text()).strip() for t in skill_tags]
            except: pass

            try:
                career_el = await page.query_selector("span:has-text('Career Level:') + span")
                if career_el: data["experience_level"] = await career_el.inner_text()
                
                type_el = await page.query_selector("span:has-text('Job Type:') + span")
                if type_el: data["employment_type"] = await type_el.inner_text()
            except: pass

        elif source == "LinkedIn":
            try:
                try: await page.click("button.show-more-less-html__button", timeout=1000)
                except: pass
                elm = await page.wait_for_selector(".show-more-less-html__markup, .description__text", timeout=3000)
                data["description"] = await elm.inner_text()
            except: data["description"] = "Description not found."
            
            try:
                apply_btn = await page.query_selector("a.top-card-layout__cta--primary, a.apply-button")
                if apply_btn:
                    href = await apply_btn.get_attribute("href")
                    if href and "linkedin.com" not in href: data["apply_link"] = href
            except: pass
            
            try:
                criteria_list = await page.query_selector_all(".description__job-criteria-item")
                for item in criteria_list:
                    header = await item.query_selector("h3")
                    val = await item.query_selector("span")
                    if header and val:
                        h_text = (await header.inner_text()).lower()
                        v_text = (await val.inner_text()).strip()
                        if "seniority" in h_text: data["experience_level"] = v_text
                        if "employment" in h_text: data["employment_type"] = v_text
            except: pass

        if data["description"]:
            reqs, resps = parse_description(data["description"])
            data["requirements"] = reqs if reqs else "See Description"
            data["responsibilities"] = resps if resps else "See Description"

    except Exception:
        pass # If detailed scrape fails, we return empty structure (soft fail)
    finally:
        await page.close()
        
    return data

# --- ASYNC SCRAPER LOGIC ---
async def run_scraper():
    joblens = JobLensScraper()
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        
        # 1. WUZZUF
        print("\n--- 1. Scraping Wuzzuf ---")
        try:
            page = await context.new_page()
            await page.route("**/*", block_resources) # Block images
            
            await page.goto("https://wuzzuf.net/search/jobs/?q=&a=hpb&filters%5Bcountry%5D%5B0%5D=Egypt", timeout=60000)
            try: await page.wait_for_selector(".css-1gatmva, div.css-pkv5jc, .css-bjn8wh", timeout=20000)
            except: print("   ⚠️ Wuzzuf selector timeout.")

            # UPDATED: Added '.css-bjn8wh' to support the specific card type you found
            cards = await page.query_selector_all(".css-1gatmva")
            if not cards: cards = await page.query_selector_all("div.css-pkv5jc")
            if not cards: cards = await page.query_selector_all("div.css-bjn8wh")
            
            print(f"   Found {len(cards)} job cards. Preparing to fetch details...")
            
            # First Pass: Extract basic info
            wuzzuf_basics = []
            for card in cards:
                try:
                    # --- UPDATED LOGIC BASED ON YOUR HTML SNIPPET ---
                    
                    # 1. Title (Try H2 first, but fallback to H1 as seen in your snippet)
                    t_el = await card.query_selector("h2")
                    if not t_el:
                         t_el = await card.query_selector("h1") # e.g. css-gkdl1m
                    title = (await t_el.inner_text()).strip() if t_el else "N/A"
                    
                    # 2. Company 
                    # STRICTLY target the class you found: 'css-p7pghv'
                    c_el = await card.query_selector("a.css-p7pghv")
                    
                    # Fallback for other card types if that specific class isn't there
                    if not c_el: c_el = await card.query_selector("a.css-17s97q8")
                    
                    company = (await c_el.inner_text()).strip().replace(" -", "") if c_el else "Confidential"

                    # 3. Location
                    # STRICTLY target the strong tag: 'css-1vlp604'
                    # Your snippet: <strong class="css-1vlp604">...Company Div... Giza, Egypt</strong>
                    l_el = await card.query_selector("strong.css-1vlp604")
                    
                    if l_el:
                        full_text = (await l_el.inner_text()).strip()
                        # The text contains "Company NameGiza, Egypt" or "Company Name - Giza, Egypt"
                        # We simply remove the company name to leave the location.
                        location = full_text.replace(company, "").replace("-", "").strip()
                    else:
                        # Fallback for old layout
                        l_old = await card.query_selector(".css-5wys0k")
                        location = (await l_old.inner_text()).strip() if l_old else "Egypt"
                    
                    # 4. Date
                    # Added 'css-154erwh' from your snippet
                    d_el = await card.query_selector(".css-do6t5g, .css-4c4ojb, .css-154erwh")
                    date_text = await d_el.inner_text() if d_el else ""
                    
                    # 5. Link
                    # Try finding link in H2 or H1
                    link_el = await card.query_selector("h2 a")
                    if not link_el: link_el = await card.query_selector("h1 a")
                    
                    # If the title is just text (no link), try finding the company link as a backup
                    post_link = ""
                    if link_el:
                         post_link = await link_el.get_attribute("href")
                    else:
                         # Sometimes the whole card is clickable or the company link is the only href
                         if c_el: post_link = await c_el.get_attribute("href")

                    wuzzuf_basics.append({
                        "source": "Wuzzuf",
                        "title": title,
                        "company": company,
                        "location": location,
                        "posted_time": parse_relative_date(date_text),
                        "job_page_link": post_link
                    })
                except Exception as e: 
                    # print(f"Error skipping card: {e}") 
                    continue

            await page.close() # Close search page
            
            # Second Pass: Batch Details Fetching
            for i in range(0, len(wuzzuf_basics), CONCURRENCY_LIMIT):
                batch = wuzzuf_basics[i : i + CONCURRENCY_LIMIT]
                print(f"   Fetching details for batch {i+1}-{i+len(batch)}...", end="\r")
                
                tasks = [get_job_details(context, item['job_page_link'], "Wuzzuf") for item in batch]
                details_list = await asyncio.gather(*tasks)
                
                # Merge details back into basic info
                for idx, details in enumerate(details_list):
                    batch[idx].update(details)
                
                joblens.save_jobs(batch)
            
            print(f"\n   ✅ Finished Wuzzuf.")

        except Exception as e: print(f"❌ Wuzzuf Error: {e}")

        # 2. LINKEDIN
        print("\n--- 2. Scraping LinkedIn ---")
        try:
            page = await context.new_page()
            await page.route("**/*", block_resources)
            
            await page.goto("https://www.linkedin.com/jobs/search?keywords=&location=Egypt&position=1&pageNum=0", timeout=60000)
            
            # Dynamic Scrolling (Smart Wait)
            print("   Scrolling to load jobs...")
            prev_count = 0
            no_change_ticks = 0
            for _ in range(15): # Max scroll attempts
                await page.mouse.wheel(0, 1500)
                await asyncio.sleep(2) # Give it time to render
                cards = await page.query_selector_all(".base-card")
                curr_count = len(cards)
                
                if curr_count == prev_count:
                    no_change_ticks += 1
                    if no_change_ticks >= 3: break # Stop if no new jobs for 3 cycles
                else:
                    no_change_ticks = 0
                prev_count = curr_count

            cards = await page.query_selector_all(".base-card")
            print(f"   Found {len(cards)} job cards. Preparing to fetch details...")

            linkedin_basics = []
            for card in cards:
                try:
                    t_el = await card.query_selector(".base-search-card__title")
                    c_el = await card.query_selector(".base-search-card__subtitle")
                    l_el = await card.query_selector(".job-search-card__location")
                    d_el = await card.query_selector("time")
                    link_el = await card.query_selector("a.base-card__full-link")
                    
                    if t_el and link_el:
                        date_attr = await d_el.get_attribute("datetime") if d_el else ""
                        if not date_attr and d_el: date_attr = (await d_el.inner_text()).strip()
                        
                        linkedin_basics.append({
                            "source": "LinkedIn",
                            "title": (await t_el.inner_text()).strip(),
                            "company": (await c_el.inner_text()).strip() if c_el else "N/A",
                            "location": (await l_el.inner_text()).strip() if l_el else "Egypt",
                            "posted_time": date_attr if date_attr else str(datetime.now().date()),
                            "job_page_link": await link_el.get_attribute("href")
                        })
                except: continue
            
            await page.close()

            # Batch Details Fetching
            for i in range(0, len(linkedin_basics), CONCURRENCY_LIMIT):
                batch = linkedin_basics[i : i + CONCURRENCY_LIMIT]
                print(f"   Fetching details for batch {i+1}-{i+len(batch)}...", end="\r")
                
                tasks = [get_job_details(context, item['job_page_link'], "LinkedIn") for item in batch]
                details_list = await asyncio.gather(*tasks)
                
                for idx, details in enumerate(details_list):
                    batch[idx].update(details)
                
                joblens.save_jobs(batch)

            print(f"\n   ✅ Finished LinkedIn.")

        except Exception as e: print(f"❌ LinkedIn Error: {e}")

        await browser.close()

d:\anaconda\envs\joblens\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Cell 3: The Execution Wrapper
import sys
from threading import Thread

def run_scraper_thread(coro):
    """
    Runs an async function in a separate thread to avoid the 
    Windows NotImplementedError in Jupyter.
    """
    def target():
        # Force the correct Windows policy inside this thread
        if sys.platform == 'win32':
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        
        # Create a fresh loop
        new_loop = asyncio.new_event_loop()
        asyncio.set_event_loop(new_loop)
        
        # Run the scraper
        try:
            new_loop.run_until_complete(coro)
        finally:
            new_loop.close()

    t = Thread(target=target)
    t.start()
    t.join()

print("🚀 Starting Optimized Smart Scraper...")
print(f"ℹ️  Configuration: Parallel Batches of 5 | Image Blocking Enabled")

# THIS REPLACES 'await run_scraper()'
run_scraper_thread(run_scraper())

print("✅ Finished.")

🚀 Starting Optimized Smart Scraper...
ℹ️  Configuration: Parallel Batches of 5 | Image Blocking Enabled
Loading AI Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 389.58it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 0 existing jobs.

--- 1. Scraping Wuzzuf ---
   Found 15 job cards. Preparing to fetch details...
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.

   ✅ Finished Wuzzuf.

--- 2. Scraping LinkedIn ---
   Scrolling to load jobs...
   Found 70 job cards. Preparing to fetch details...
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for database...
   ✅ Success: Added 5 NEW jobs to DB.
   Processing 5 candidates for 

In [4]:
# Cell 4: Verify Database Content
import chromadb
import json
import random  # Added for random selection

# 1. Connect to the SAME database path used in your scraper
CHROMA_PATH = "./joblens_db"
client = chromadb.PersistentClient(path=CHROMA_PATH)

# 2. Get the collection
try:
    collection = client.get_collection(name="job_listings")
    
    # 3. Check the Count
    count = collection.count()
    print(f"✅ CONNECTION SUCCESSFUL")
    print(f"📊 Total Jobs in Database: {count}")

    if count > 0:
        # 4. Pick 3 Random Jobs to Inspect
        print("\n🔍 Inspecting 3 RANDOM jobs:")
        
        # Get all IDs first to select randomly
        all_ids = collection.get()['ids']
        
        # Determine sample size (3 or fewer if the DB has less than 3 items)
        sample_size = min(3, len(all_ids))
        
        # Pick random IDs
        random_ids = random.sample(all_ids, sample_size)
        
        # Fetch the specific data for these random IDs
        data = collection.get(ids=random_ids)
        
        # 'data' is a dictionary of lists. We iterate through them.
        for i in range(len(data['ids'])):
            print("-" * 40)
            print(f"ID:     {data['ids'][i]}")
            print(f"Source: {data['metadatas'][i].get('source', 'N/A')}")
            print(f"Title:  {data['metadatas'][i].get('title', 'N/A')}")
            print(f"Company:{data['metadatas'][i].get('company', 'N/A')}")
            
            # FIXED: Use 'job_page_link' instead of 'link'
            print(f"Link:   {data['metadatas'][i].get('job_page_link', 'N/A')}")
            
            # Optional: Verify the JSON detail is valid
            try:
                details = json.loads(data['metadatas'][i]['json_detailed'])
                print(f"JSON Check: Valid ✅ (Location: {details.get('location', 'N/A')})")
            except:
                print("JSON Check: Invalid ❌")
    else:
        print("\n⚠️ The database is empty! Run 'smart_scraper.py' first.")

except Exception as e:
    print(f"❌ Error: Could not load collection. Reason: {e}")

✅ CONNECTION SUCCESSFUL
📊 Total Jobs in Database: 73

🔍 Inspecting 3 RANDOM jobs:
----------------------------------------
ID:     98425da67a234bc8a86397e2b289b14e
Source: Wuzzuf
Title:  Call Center Advisor (B1+ - B2 English Speaker)|E&UAE
Company:Confidential
Link:   https://wuzzuf.net/jobs/p/frqtertyah46-call-center-advisor-b1-b2-english-speakereuae-etisalat-egypt-cairo-egypt
JSON Check: Valid ✅ (Location: Egypt)
----------------------------------------
ID:     c628492b1d1dc35a96a61bd58ad8ead6
Source: LinkedIn
Title:  Alexandria Cabin Crew Opportunities (Dubai Based, Relocation Provided)
Company:Emirates
Link:   https://eg.linkedin.com/jobs/view/alexandria-cabin-crew-opportunities-dubai-based-relocation-provided-at-emirates-4356302544?position=11&pageNum=0&refId=Fi6z1HCSTmExgZzGsCxa%2Bg%3D%3D&trackingId=3tJe2Pjp0uFap%2FwqugU8Tw%3D%3D
JSON Check: Valid ✅ (Location: Alexandria, Alexandria, Egypt)
----------------------------------------
ID:     2364717d6a8899aeba00a268a05a0e77
Source: 

In [5]:
# Cell 5: JobLens API (Complete: Search, Scraper, Create, Update, Delete)
import uvicorn
import uuid
import json
import datetime
import re
from fastapi import FastAPI, HTTPException, BackgroundTasks, Query
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
import chromadb
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
CHROMA_PATH = "./joblens_db"
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

# --- APP SETUP ---
app = FastAPI(title="JobLens API")

print("⏳ Loading AI Model & Database...")
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name="job_listings")
model = SentenceTransformer(EMBEDDING_MODEL)
print("✅ System Ready.")

# --- MODELS ---
class JobData(BaseModel):
    title: str
    description: str
    requirements: Optional[str] = "Not Specified"
    responsibilities: Optional[str] = "Not Specified"
    skills: List[str] = []
    location: str
    experience_level: str = "Not Specified"
    employment_type: str = "Not Specified"
    company_name: str

class JobEmbeddingRequest(BaseModel):
    job_id: int
    job_data: JobData

class ScrapingTriggerRequest(BaseModel):
    sources: List[str] = ["linkedin", "wuzzuf"]
    keywords: List[str] = []
    locations: List[str] = ["Egypt"]
    max_pages: int = 1
    categories: Optional[Dict[str, List[str]]] = None

# --- HELPER: BACKGROUND TASK ---
async def run_background_scraper(params: ScrapingTriggerRequest):
    print(f"🔄 BACKGROUND TASK: Scraping {getattr(params, 'keywords', [])} categories={getattr(params, 'categories', {})}...")
    await run_scraper(params)
    print("✅ BACKGROUND TASK FINISHED.")

# --- HELPER: PROCESS JOB ---
def process_and_store_job(job_id: int, job_data: JobData):
    skills_str = ", ".join(job_data.skills)
    full_text = (
        f"{job_data.title} at {job_data.company_name}. "
        f"Location: {job_data.location}. "
        f"Level: {job_data.experience_level}. "
        f"Skills: {skills_str}. "
        f"Description: {job_data.description} "
        f"Requirements: {job_data.requirements}"
    )

    embedding = model.encode(full_text).tolist()
    embedding_id = f"job_{job_id}"
    
    # Save full details for retrieval
    full_job_dict = {
        "title": job_data.title,
        "company": job_data.company_name,
        "location": job_data.location,
        "description": job_data.description,
        "requirements": job_data.requirements,
        "responsibilities": job_data.responsibilities,
        "skills": job_data.skills,
        "experience_level": job_data.experience_level,
        "employment_type": job_data.employment_type
    }

    metadata = {
        "original_job_id": job_id,
        "title": job_data.title,
        "company": job_data.company_name, # Saved here so we can retrieve it
        "location": job_data.location,
        "source": "Internal API",
        "json_detailed": json.dumps(full_job_dict)
    }

    collection.upsert(
        ids=[embedding_id],
        embeddings=[embedding],
        documents=[full_text],
        metadatas=[metadata]
    )
    return embedding_id

# --- ENDPOINTS ---

# 1. GET SCRAPED JOBS (Returns Company in Response)
@app.get("/api/scraping/jobs")
async def get_scraped_jobs(
    keyword: Optional[str] = Query(None),
    location: Optional[str] = Query(None),
    limit: int = 50
):
    try:
        where_filter = {}
        if location: where_filter["location"] = location
        if len(where_filter) == 0: where_filter = None

        if keyword:
            # Semantic Search
            query_vec = model.encode(keyword).tolist()
            results = collection.query(
                query_embeddings=[query_vec], n_results=limit, where=where_filter
            )
            ids = results['ids'][0]
            metas = results['metadatas'][0]
        else:
            # List All
            results = collection.get(limit=limit, where=where_filter)
            ids = results['ids']
            metas = results['metadatas']

        formatted_jobs = []
        for i in range(len(ids)):
            m = metas[i]
            
            # Load extra details if they exist
            full_details = {}
            if "json_detailed" in m:
                try: full_details = json.loads(m["json_detailed"])
                except: full_details = {}

            # Build the response object explicitly including "company"
            job_response = {
                "db_id": ids[i],
                # This grabs 'company' from the metadata we saved in ChromaDB
                "company": m.get("company", "N/A"), 
                "title": m.get("title", "N/A"),
                "location": m.get("location", "N/A"),
                
                # Merge in the rest of the scraped details (skills, links, etc.)
                **full_details 
            }
            
            formatted_jobs.append(job_response)
            
        return {"success": True, "count": len(ids), "data": formatted_jobs}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 2. TRIGGER SCRAPING
@app.post("/api/scraping/trigger", status_code=202)
async def trigger_scraping_job(payload: ScrapingTriggerRequest, bt: BackgroundTasks):
    bt.add_task(run_background_scraper, payload)
    return {"success": True, "message": "Scraping queued"}

# 3. CREATE EMBEDDING
@app.post("/api/embeddings/job")
async def create_job_embedding(payload: JobEmbeddingRequest):
    try:
        eid = process_and_store_job(payload.job_id, payload.job_data)
        return {"success": True, "message": "Created", "data": {"id": eid}}
    except Exception as e: raise HTTPException(status_code=500, detail=str(e))

# 4. UPDATE EMBEDDING
@app.put("/api/embeddings/job/{job_id}")
async def update_job_embedding(job_id: int, payload: JobEmbeddingRequest):
    try:
        if job_id != payload.job_id:
            raise HTTPException(status_code=400, detail="URL ID mismatch")
        
        process_and_store_job(payload.job_id, payload.job_data)
        return {"success": True, "message": "Job embedding updated successfully"}
    except Exception as e: raise HTTPException(status_code=500, detail=str(e))

# 5. DELETE EMBEDDING
@app.delete("/api/embeddings/job/{job_id}")
async def delete_job_embedding(job_id: str, force: bool = Query(False)):
    """
    Deletes embeddings by flexible identifiers:
      - 'job_<number>' (direct id)
      - numeric string '42' (tries 'job_42' then metadata lookup original_job_id)
      - md5 hex id for scraped entries
      - literal id fallback
    Use ?force=true to confirm deletion when multiple matches are found.
    """
    try:
        # 1) Direct 'job_<n>' (common for API-created entries)
        if re.match(r"^job_\d+$", job_id):
            res = collection.get(ids=[job_id])
            if res.get('ids'):
                collection.delete(ids=[job_id])
                return {"success": True, "deleted_ids": [job_id]}
            raise HTTPException(status_code=404, detail="ID not found")

        # 2) Numeric ID (e.g., '42') -> try 'job_42' then metadata lookup
        if re.match(r"^\d+$", job_id):
            candidate = f"job_{job_id}"
            res = collection.get(ids=[candidate])
            if res.get('ids'):
                collection.delete(ids=[candidate])
                return {"success": True, "deleted_ids": [candidate]}

            # Lookup by metadata original_job_id
            try:
                res = collection.get(where={"original_job_id": int(job_id)})
            except Exception:
                res = collection.get(where={"original_job_id": str(job_id)})
            ids = res.get('ids', [])
            if not ids:
                raise HTTPException(status_code=404, detail="No matching entries found for that job id")

            if len(ids) > 1 and not force:
                raise HTTPException(status_code=409, detail=f"Multiple matches found: {ids}. Re-run with ?force=true to delete all.")
            collection.delete(ids=ids)
            return {"success": True, "deleted_ids": ids}

        # 3) MD5 hex (scraped id)
        if re.match(r"^[a-f0-9]{32}$", job_id):
            res = collection.get(ids=[job_id])
            if res.get('ids'):
                collection.delete(ids=[job_id])
                return {"success": True, "deleted_ids": [job_id]}
            raise HTTPException(status_code=404, detail="ID not found")

        # 4) Fallback: try literal id
        res = collection.get(ids=[job_id])
        if res.get('ids'):
            collection.delete(ids=[job_id])
            return {"success": True, "deleted_ids": [job_id]}

        raise HTTPException(status_code=404, detail="ID not found")
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- EXECUTION ---
print("🚀 JobLens API running at http://127.0.0.1:8000")
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
await server.serve()

⏳ Loading AI Model & Database...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 385.23it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ System Ready.
🚀 JobLens API running at http://127.0.0.1:8000
